# `pygdina` Demo

- 2026-04-09
- jwt

經過各種測試改寫後，這個版本應該是 OK.
使用「正確識別」的 Q-matrix 與資料，可以確保在 R-GDINA 和本程式的估計結果是一致的。

（NOTE. 沒有正確識別的 Q matrix 仍會有和 R-GDINA 不一致的情形。但那是因為沒有正確識別而導致的估計問題，和本程式無關。）

## 使用方式

依照以下方式用 `!wget` 下載後，像一般套件一樣用 `from pygdina import GDINA`.

範例的 data 及 Q 也有提供。（從 R-GDINA 中的 sim30GDINA 而來）。

In [1]:
# 1. 刪除所有舊的 pygdina 相關檔案
#!rm -f pygdina.py*

# 2. 下載最新的版本（確保檔名是 pygdina.py）
!wget https://raw.githubusercontent.com/jiewenTsai/pygdina/refs/heads/main/pygdina.py

--2026-04-10 04:10:56--  https://raw.githubusercontent.com/jiewenTsai/pygdina/refs/heads/main/pygdina.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 28326 (28K) [text/plain]
Saving to: ‘pygdina.py’

pygdina.py          100%[===================>]  27.66K  --.-KB/s    in 0.001s  

2026-04-10 04:10:57 (27.8 MB/s) - ‘pygdina.py’ saved [28326/28326]



In [2]:
import numpy as np
#from itertools import combinations
#from scipy.special import logsumexp
#from numba import njit
import pandas as pd
from pygdina import GDINA, check_q_identifiability

In [3]:
data = pd.read_csv('https://raw.githubusercontent.com/jiewenTsai/pygdina/main/simData.csv')
Q = pd.read_csv('https://raw.githubusercontent.com/jiewenTsai/pygdina/main/simQ.csv')

In [5]:
# 檢查你的 Q 矩陣
report = check_q_identifiability(Q)
print(f"辨識等級: {report['level']}")

辨識等級: 強可辨識 (Strongly Identifiable)


In [4]:
gdina = GDINA()
gdina.fit(data, Q)

  Start 1 (R-default): deviance = 38881.54
  Start 2 (DINA): deviance = 45930.80
  Start 3 (DINO): deviance = 45738.17
  → selected Start 1

Iter=  33  |ΔLL|=8.26e-06  deviance=33999.1731
converged  deviance=33999.1731  iter=33


In [6]:
pd.DataFrame(gdina.person_parm('mp')).round(4).head(30)

,0,1,2,3,4
0,0.7396,0.8009,0.9958,0.0316,0.9055
1,0.8637,0.9996,0.0000,0.9569,0.9881
2,0.5390,0.9882,0.0006,0.9924,0.9919
3,0.9985,0.9905,0.0001,0.9989,0.9490
4,0.0014,0.0002,1.0000,0.0054,0.9998
5,0.0030,0.0083,0.0053,0.1358,0.9901
6,0.0059,0.6072,0.9919,0.0016,0.9794
7,0.8863,0.9582,0.3312,0.0046,0.1204
8,0.0208,0.9861,1.0000,0.0004,0.9999
9,0.9773,0.9935,0.9913,0.9999,1.0000


In [9]:
pd.DataFrame(gdina.item_table()).round(4)

,item,group,pattern,P
0,1,1,"(0,)",0.0868
1,1,2,"(1,)",0.7665
2,2,1,"(0,)",0.1306
3,2,2,"(1,)",0.7858
4,3,1,"(0,)",0.0820
...,...,...,...,...
135,30,4,"(1, 1, 0)",0.5705
136,30,5,"(0, 0, 1)",0.4219
137,30,6,"(1, 0, 1)",0.6413
138,30,7,"(0, 1, 1)",0.6115


In [ ]:
pd.DataFrame(gdina.item_parm).round(4)

,0,1,2,3,4,5,6,7
0,0.0868,0.7665,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
1,0.1306,0.7858,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
2,0.0820,0.7958,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
3,0.1097,0.8117,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
4,0.0885,0.8017,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
5,0.1831,0.9004,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
6,0.1730,0.9073,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
7,0.1888,0.8958,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
8,0.2104,0.8857,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
9,0.1937,0.8990,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000


## 和 R-GDINA 比較

In [ ]:
%load_ext rpy2.ipython

In [ ]:
%%R
system('sudo apt upgrade')
system('sudo apt install r-cran-gdina')

In [ ]:
%%R
library(GDINA)
sim30GDINA$simdat |> head()


     [,1] [,2] [,3] [,4] [,5] [,6] [,7] [,8] [,9] [,10] [,11] [,12] [,13] [,14]
[1,]    1    0    1    0    1    0    1    1    0     1     1     1     0     1
[2,]    0    1    0    1    1    1    1    0    0     1     1     0     1     0
[3,]    1    1    0    0    1    0    1    0    1     1     0     1     1     1
[4,]    1    0    0    1    1    1    1    0    1     0     1     0     1     1
[5,]    0    0    1    0    1    0    0    1    0     1     0     1     0     0
[6,]    0    0    0    0    1    0    0    1    1     1     0     0     0     1
     [,15] [,16] [,17] [,18] [,19] [,20] [,21] [,22] [,23] [,24] [,25] [,26]
[1,]     1     1     0     0     0     0     0     1     1     1     1     0
[2,]     0     1     1     0     0     0     1     1     1     1     1     1
[3,]     0     1     0     0     0     1     1     1     1     1     1     0
[4,]     0     1     1     0     0     0     1     1     1     0     1     1
[5,]     0     0     0     0     1     1     1     0   

In [ ]:
%%R
sim30GDINA$simQ

   V1 V2 V3 V4 V5
1   1  0  0  0  0
2   0  1  0  0  0
3   0  0  1  0  0
4   0  0  0  1  0
5   0  0  0  0  1
6   1  0  0  0  0
7   0  1  0  0  0
8   0  0  1  0  0
9   0  0  0  1  0
10  0  0  0  0  1
11  1  1  0  0  0
12  1  0  1  0  0
13  1  0  0  1  0
14  1  0  0  0  1
15  0  1  1  0  0
16  0  1  0  1  0
17  0  1  0  0  1
18  0  0  1  1  0
19  0  0  1  0  1
20  0  0  0  1  1
21  1  1  1  0  0
22  1  1  0  1  0
23  1  1  0  0  1
24  1  0  1  1  0
25  1  0  1  0  1
26  1  0  0  1  1
27  0  1  1  1  0
28  0  1  1  0  1
29  0  1  0  1  1
30  0  0  1  1  1


In [ ]:
%%R
fit <- GDINA(dat = sim30GDINA$simdat, Q = sim30GDINA$simQ)

Iter = 23  Max. abs. change = 0.00009  Deviance  = 33999.17                                                                                  

以下兩個估計結果，可以和前面的 pygdina 估計結果作比較。應該基本是一樣的。

In [ ]:
%%R
personparm(fit, 'mp') |> head(30)

          A1     A2     A3     A4     A5
 [1,] 0.7405 0.8008 0.9958 0.0315 0.9055
 [2,] 0.8642 0.9996 0.0000 0.9568 0.9881
 [3,] 0.5405 0.9881 0.0006 0.9924 0.9919
 [4,] 0.9985 0.9905 0.0001 0.9989 0.9490
 [5,] 0.0014 0.0002 1.0000 0.0054 0.9998
 [6,] 0.0031 0.0083 0.0053 0.1359 0.9901
 [7,] 0.0059 0.6074 0.9919 0.0016 0.9794
 [8,] 0.8868 0.9582 0.3311 0.0046 0.1204
 [9,] 0.0208 0.9861 1.0000 0.0004 0.9999
[10,] 0.9774 0.9935 0.9913 0.9999 1.0000
[11,] 0.9989 0.5097 0.2813 0.2041 0.0026
[12,] 0.0004 0.0004 0.0436 0.8858 0.0428
[13,] 0.0069 0.9999 0.9997 0.0279 0.9999
[14,] 0.0001 0.0002 0.0028 0.0003 0.0104
[15,] 0.0018 0.9940 0.0172 0.0058 0.9998
[16,] 0.0063 0.0001 0.9512 0.0003 0.9980
[17,] 0.0687 0.9954 0.0009 0.0005 0.9995
[18,] 0.0440 0.9996 0.0076 0.1446 0.9987
[19,] 0.0025 0.0124 0.0001 0.1216 0.9972
[20,] 0.6764 0.9945 0.9997 0.9993 0.9990
[21,] 0.8836 0.0083 0.1700 0.2584 0.9854
[22,] 0.0071 0.9941 0.9999 0.0003 0.0033
[23,] 0.1913 0.9839 0.7911 0.9968 0.9995
[24,] 0.5709 0.7

In [ ]:
%%R
coef(fit)

$`Item 1`
  P(0)   P(1) 
0.0867 0.7663 

$`Item 2`
  P(0)   P(1) 
0.1306 0.7859 

$`Item 3`
  P(0)   P(1) 
0.0820 0.7958 

$`Item 4`
  P(0)   P(1) 
0.1097 0.8117 

$`Item 5`
  P(0)   P(1) 
0.0885 0.8017 

$`Item 6`
  P(0)   P(1) 
0.1828 0.9003 

$`Item 7`
  P(0)   P(1) 
0.1731 0.9073 

$`Item 8`
  P(0)   P(1) 
0.1888 0.8958 

$`Item 9`
  P(0)   P(1) 
0.2104 0.8858 

$`Item 10`
  P(0)   P(1) 
0.1937 0.8990 

$`Item 11`
 P(00)  P(10)  P(01)  P(11) 
0.2738 0.5579 0.4926 0.7745 

$`Item 12`
 P(00)  P(10)  P(01)  P(11) 
0.2269 0.5218 0.4790 0.7519 

$`Item 13`
 P(00)  P(10)  P(01)  P(11) 
0.2038 0.4733 0.5401 0.7913 

$`Item 14`
 P(00)  P(10)  P(01)  P(11) 
0.2250 0.4642 0.5164 0.8038 

$`Item 15`
 P(00)  P(10)  P(01)  P(11) 
0.1242 0.0975 0.1334 0.7735 

$`Item 16`
 P(00)  P(10)  P(01)  P(11) 
0.0913 0.0885 0.0873 0.8394 

$`Item 17`
 P(00)  P(10)  P(01)  P(11) 
0.1271 0.1067 0.1063 0.8495 

$`Item 18`
 P(00)  P(10)  P(01)  P(11) 
0.1156 0.0657 0.0940 0.7866 

$`Item 19`
 P(00)  P(10)  P(0

## Simulation


In [ ]:
import numpy as np
import time
from itertools import combinations as icombs
from pygdina import GDINA, _skill_profiles

# ============================================================
# 0. 基本設定
# ============================================================
N, J, K = 1000, 20, 5
N_SIMS = 20  # 模擬次數
TRUE_MARG = np.array([0.60, 0.50, 0.70, 0.40, 0.65])
ap_full = _skill_profiles(K)

# 設定一個固定數字，任何數字都可以
np.random.seed(1234)

# 用於儲存每次模擬結果的清單
results_item_rmse = []
results_emr = []
results_prior_mae = []
results_skill_corr = [] # 儲存 A1~A5 的平均相關係數

# ============================================================
# 1. 定義輔助函數
# ============================================================
def _build_Mj(Kj: int) -> np.ndarray:
    ap = _skill_profiles(Kj)
    col_defs = [()]
    for r in range(1, Kj + 1):
        for combo in icombs(range(Kj), r): col_defs.append(combo)
    M = np.zeros((2 ** Kj, len(col_defs)))
    for c, col in enumerate(col_defs):
        M[:, c] = 1.0 if not col else np.prod(ap[:, list(col)], axis=1)
    return M

# ============================================================
# 2. 開始重複模擬迴圈
# ============================================================
print(f"開始執行 {N_SIMS} 次模擬研究 (N={N}, J={J}, K={K})...\n")
start_time = time.time()

for s_idx in range(N_SIMS):
    # 每次模擬使用不同的隨機種子以確保獨立性
    current_seed = 2024 + s_idx
    np.random.seed(current_seed)

    # --- A. 生成 Q-matrix 與 真實題參數 ---
    Q = np.zeros((J, K))
    for k in range(K): Q[k, k] = 1
    for j in range(5, 15): Q[j, np.random.choice(K, 2, replace=False)] = 1
    for j in range(15, 20): Q[j, np.random.choice(K, 3, replace=False)] = 1

    true_P = []
    for j in range(J):
        Kj = int(Q[j].sum())
        M = _build_Mj(Kj)
        g, s = np.random.uniform(0.05, 0.20, 2)
        delta = np.concatenate([[g], np.random.dirichlet(np.ones(M.shape[1]-1)) * ((1-s)-g)])
        true_P.append(np.clip(M @ delta, 0.001, 0.999))

    # --- B. 生成受測者與反應資料 ---
    true_lp = ap_full @ np.log(TRUE_MARG) + (1-ap_full) @ np.log(1-TRUE_MARG)
    true_prior = np.exp(true_lp) / np.sum(np.exp(true_lp))
    true_cls = np.random.choice(len(ap_full), size=N, p=true_prior)
    true_pers = ap_full[true_cls]

    temp_mod = GDINA(verbose=False)
    temp_mod.att_pattern = ap_full
    parloc = temp_mod._build_parloc(Q)

    Y = np.zeros((N, J), dtype=float)
    for j in range(J):
        Y[:, j] = np.random.binomial(1, true_P[j][parloc[j, true_cls] - 1])

    # --- C. 模型估計 ---
    model = GDINA(max_iter=500, verbose=False).fit(Y, Q)

    # --- D. 計算並記錄指標 ---
    # 1. Item RMSE
    rmses = [np.sqrt(np.mean((model.item_parm[j, :len(true_P[j])] - true_P[j])**2)) for j in range(J)]
    results_item_rmse.append(np.mean(rmses))

    # 2. Exact Match Ratio (EMR)
    eap = model.person_parm("eap")
    results_emr.append(np.all(true_pers == eap, axis=1).mean())

    # 3. Prior MAE
    est_prior = np.exp(model.log_prior)
    results_prior_mae.append(np.abs(est_prior - true_prior).mean())

    # 4. Skill Correlation (平均 5 個屬性的相關)
    mp = model.person_parm("mp")
    corrs = [np.corrcoef(true_pers[:, k], mp[:, k])[0, 1] for k in range(K)]
    results_skill_corr.append(np.mean(corrs))

    print(f"Simulation {s_idx+1}/{N_SIMS} 完成 (EMR: {results_emr[-1]:.3f})")

# ============================================================
# 3. 輸出最終統計結果
# ============================================================
total_elapsed = time.time() - start_time
print("\n" + "═"*60)
print(f"模擬研究總結 ({N_SIMS} 次重複迭代)")
print(f"總耗時: {total_elapsed:.1f} 秒")
print("-" * 60)
print(f"{'指標名稱 (Metric)':<30} | {'平均值 (Mean)':>10} | {'標準差 (SD)':>10}")
print("-" * 60)
metrics = [
    ("Mean Item RMSE", results_item_rmse),
    ("Exact Match Ratio (EMR)", results_emr),
    ("Prior Distribution MAE", results_prior_mae),
    ("Average Skill Correlation", results_skill_corr)
]

for name, data in metrics:
    print(f"{name:<30} | {np.mean(data):10.4f} | {np.std(data):10.4f}")
print("═"*60)

開始執行 20 次模擬研究 (N=1000, J=20, K=5)...

Simulation 1/20 完成 (EMR: 0.667)
Simulation 2/20 完成 (EMR: 0.726)
Simulation 3/20 完成 (EMR: 0.720)
Simulation 4/20 完成 (EMR: 0.711)
Simulation 5/20 完成 (EMR: 0.687)
Simulation 6/20 完成 (EMR: 0.765)
Simulation 7/20 完成 (EMR: 0.698)
Simulation 8/20 完成 (EMR: 0.695)
Simulation 9/20 完成 (EMR: 0.723)
Simulation 10/20 完成 (EMR: 0.582)
Simulation 11/20 完成 (EMR: 0.751)
Simulation 12/20 完成 (EMR: 0.646)
Simulation 13/20 完成 (EMR: 0.729)
Simulation 14/20 完成 (EMR: 0.697)
Simulation 15/20 完成 (EMR: 0.738)
Simulation 16/20 完成 (EMR: 0.709)
Simulation 17/20 完成 (EMR: 0.688)
Simulation 18/20 完成 (EMR: 0.654)
Simulation 19/20 完成 (EMR: 0.698)
Simulation 20/20 完成 (EMR: 0.754)

════════════════════════════════════════════════════════════
模擬研究總結 (20 次重複迭代)
總耗時: 18.1 秒
------------------------------------------------------------
指標名稱 (Metric)                  | 平均值 (Mean) |   標準差 (SD)
------------------------------------------------------------
Mean Item RMSE                 |     0.0

本次模擬研究顯示 **GDINA 模型引擎具備優異的參數恢復性**，在 20 次重複迭代中，題參數平均 RMSE 僅為 0.0354，展現極高的估計精準度。受測者分類能力穩定，**平均屬性相關達 0.8777** 且全剖面一致率（EMR）突破七成，能有效區分受測者的潛在能力狀態。整體而言，該演算法在 Google Colab 環境下運算效能極佳（20 次模擬僅耗時 18.3 秒），**足以支援大規模的認知診斷研究與資料分析任務**。

## Q matrix 可識別性檢查

這部分就是需要檢查 Q 是否具可識別性。

Q-matrix 的識別性是一種「唯一性」的檢查。如果一個 Q-matrix 缺乏識別性（限制條件不夠），就會發生不同的屬性組合（profiles）在測驗中得到完全一樣的作答反應；這就像兩把不同的鑰匙都能開同一把鎖，我們就沒辦法從「鎖開了」去反推學生到底是哪一種能力狀態。

DCM 比起 IRT 或一般的 HLM 更強調這一點，是因為 DCM 是把能力切成「離散」的組合。在連續空間的模型裡，頂多是估計得準不準的問題，但在 DCM 裡這卻是「全或無」的結構問題——只要 Q-matrix 設計有缺陷，不管收再多學生資料都救不回來，DCM 模型在數學上就是分不出那些「無法區分」的屬性組合 profiles。

因此，確保 Q-matrix 具備完備性（Completeness）是診斷的前提。只有當每一種屬性組合都能對應到唯一的作答模式時，模型才具備「反推」的條件，否則這種「多對一」的錯誤映射，會讓診斷學生精熟剖面的功能失效。

**參考文獻：**
* **Chiu, C. Y., Douglas, J. A., & Li, X. (2009).** *Cluster analysis for cognitive diagnosis: Theory and applications.* (強調了完備性條件是區分所有潛在類別的關鍵)。
* **de la Torre, J. (2011).** *The generalized DINA model framework.* (探討了參數識別性與模型結構穩定性的關係)。

In [ ]:
Q0 = pd.read_excel('https://www.dropbox.com/scl/fi/6nbrifq2y2i64jwtds9ui/concepts.xlsx?rlkey=z5gs5xy5jrspoh4kkwxja5y91&e=1&dl=1')
Q1 = Q0.iloc[:,1:]

In [ ]:
Q1

,基礎性綜合應用,理論應用,程序與理論應用,計算程序型,高階概念與可衍伸性
0,0,1,0,0,1
1,1,0,0,0,1
2,0,0,0,0,1
3,1,0,0,0,1
4,0,0,0,1,1
5,0,0,1,1,1
6,0,1,0,1,1
7,0,0,0,1,1
8,0,0,0,1,0
9,0,0,1,1,1


In [ ]:
check_q_identifiability(Q1)

{'is_identifiable': False,
 'level': '不可辨識 (Unidentifiable)',
 'details': '【結構缺失】不同的屬性剖面產生完全相同的作答模式。',
 'consequence': '後果：模型參數無唯一解，EM 演算法將無法收斂或隨機收斂，分類結果 (EMR) 將失去心理計量學意義。',
 'aliased_profiles': [[[0, 0, 0, 0, 0],
   [1, 0, 0, 0, 0],
   [0, 1, 0, 0, 0],
   [1, 1, 0, 0, 0],
   [0, 0, 1, 0, 0],
   [1, 0, 1, 0, 0],
   [0, 1, 1, 0, 0],
   [1, 1, 1, 0, 0]],
  [[0, 0, 0, 1, 0],
   [1, 0, 0, 1, 0],
   [0, 1, 0, 1, 0],
   [1, 1, 0, 1, 0],
   [0, 0, 1, 1, 0],
   [1, 0, 1, 1, 0],
   [0, 1, 1, 1, 0],
   [1, 1, 1, 1, 0]],
  [[0, 0, 0, 0, 1], [0, 0, 1, 0, 1]],
  [[1, 0, 0, 0, 1], [1, 0, 1, 0, 1]],
  [[0, 1, 0, 0, 1], [0, 1, 1, 0, 1]],
  [[1, 1, 0, 0, 1], [1, 1, 1, 0, 1]]]}

In [ ]:
Q

array([[1., 0., 0., 0., 0.],
       [0., 1., 0., 0., 0.],
       [0., 0., 1., 0., 0.],
       [0., 0., 0., 1., 0.],
       [0., 0., 0., 0., 1.],
       [0., 0., 0., 1., 1.],
       [0., 0., 1., 1., 0.],
       [0., 1., 0., 1., 0.],
       [0., 0., 1., 0., 1.],
       [0., 0., 0., 1., 1.],
       [1., 1., 0., 0., 0.],
       [0., 0., 1., 1., 0.],
       [0., 1., 1., 0., 0.],
       [0., 1., 1., 0., 0.],
       [0., 1., 0., 1., 0.],
       [1., 0., 0., 1., 1.],
       [1., 1., 1., 0., 0.],
       [0., 0., 1., 1., 1.],
       [1., 0., 0., 1., 1.],
       [1., 1., 1., 0., 0.]])

In [ ]:
check_q_identifiability(Q)

{'is_identifiable': True,
 'level': '強可辨識 (Strongly Identifiable)',
 'details': '【最優結構】滿足 Chiu et al. (2009) 完備性條件 (Completeness Condition)。',
 'consequence': '後果：具備最強的統計檢定力，EM 演算法收斂速度最快，且分類準確度最為穩定。',
 'aliased_profiles': []}